In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from shapely.ops import polygonize
import json
import geopandas as gpd
import topojson as tp
from shapely.validation import make_valid
from shapely.geometry import Point

In [ ]:
import warnings
# 필터링 실행 줄보다 '위쪽(먼저)'에 위치해야 함
warnings.filterwarnings("ignore", message=".*GeoSeries\\.notna.*", category=UserWarning)

In [ ]:
DATA_DIR = "/Library/WebServer/Documents/data"
file_path = [
    f"{DATA_DIR}/earth-topo.json",
    f"{DATA_DIR}/download/rtofs_glo_2ds_f000_ice.nc",
    f"{DATA_DIR}/download/rtofs_glo_2ds_f000_prog.nc"
]

1. earth-topo.json 로드 및 육지(Land) 폴리곤 생성
- earth-topo.json의 coastline_50m은 LineString이므로 폴리곤화(polygonize) 작업이 필요합니다.
- TopoJSON -> GeoDataFrame 변환 (해상도에 따라 'coastline_50m' 또는 'coastline_110m' 선택)
- coastline_50m와 coastline_110m은 해안선(육지/해안 경계) 데이터이고, lakes_50m와 lakes_110m는 호수 데이터입니다.
- 육지 판정(contains / intersects 등) 목적이라면 해안선 레이어를 고르되,
선(LineString)인지 폴리곤(Polygon / MultiPolygon)인지 지오메트리 타입도 한 번 확인해 보세요.
- 해안선(LineString)들을 닫힌 다각형(Polygon, 즉 육지 영역)으로 변환
- earth-topo.json은 50m(1:50,000,000) 축척 데이터인 반면 RTOFS 글로벌 모델은 약 1/12°(약 8~9km) 해상도를 가집니다.
- 따라서 좁은 해협, 피오르드, 작은 섬 부근의 연안 격자는 50m 축척 폴리곤과 미세한 경계 차이로 인해 육지 내부로 판정될 수 있습니다.
- 연안 오차를 감안하려면 land_gdf.geometry = land_gdf.geometry.buffer(-0.05)처럼 육지 경계를 안쪽으로 살짝 축소하여 테스트하는 것이 좋습니다.

In [ ]:
with open(file_path[0], "r", encoding="utf-8") as f:
    topo_data = json.load(f)

available_objects = list(topo_data.get("objects", {}).keys())
available_objects

In [ ]:
polygon_layers: tuple[str, ...] = ("countries1", "countries2", "land1", "land2", "land3", "land4")
polygon_layers

In [ ]:
def _clean_gdf_4326(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """지오메트리를 유효화하고 EPSG:4326을 세팅합니다."""
    if gdf is None or gdf.empty:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry").set_crs("EPSG:4326", allow_override=True)

    out = gdf[["geometry"]].copy()
    out["geometry"] = out.geometry.apply(make_valid)
    out = out[out.geometry.notna() & (~out.geometry.is_empty) & out.geometry.is_valid].copy()
    return gpd.GeoDataFrame(out, geometry="geometry").set_crs("EPSG:4326", allow_override=True)


In [ ]:
gdfs: list[gpd.GeoDataFrame] = []
for obj_name in polygon_layers:
    if obj_name in available_objects:
        gdf = tp.Topology(topo_data, object_name=obj_name).to_gdf()
        #if not gdf.empty:
            #gdfs.append(gdf)
        cleaned = _clean_gdf_4326(gdf)
        if not cleaned.empty:
            gdfs.append(cleaned)

gdfs

In [ ]:
merged = pd.concat(gdfs, ignore_index=True)
land_gdf = _clean_gdf_4326(merged)
land_gdf

In [ ]:
buffer_meters: float = 0.0
if buffer_meters != 0.0 and not land_gdf.empty:
    land_proj = land_gdf.to_crs("ESRI:54035")
    land_proj["geometry"] = land_proj.geometry.buffer(buffer_meters)
    land_gdf = _clean_gdf_4326(land_proj.to_crs("EPSG:4326"))

In [ ]:
land_gdf_4326 = land_gdf[["geometry"]].reset_index(drop=True)

In [ ]:
def normalize_lon_180(lon: float) -> float:
    """경도를 [-180, 180] 범위로 정규화합니다."""
    lon_360 = lon % 360.0
    return ((lon_360 + 180.0) % 360.0) - 180.0


def is_on_land(lat: float, lon: float, land_gdf: gpd.GeoDataFrame) -> bool:
    """주어진 (lat, lon)이 육지 폴리곤 내부에 포함되면 True, 아니면 False를 반환합니다."""
    pt = Point(normalize_lon_180(lon), lat)
    return bool(land_gdf.geometry.contains(pt).any())


def validate_land_gdf(land_gdf: gpd.GeoDataFrame, test_cases: list[tuple[str, float, float, bool]]) -> None:
    """육지 폴리곤 GeoDataFrame과 기준 좌표 판정 결과를 검증합니다."""
    print("=== 1. 폴리곤 레이어 검증 ===")
    print(f"CRS: {land_gdf.crs}")
    print(f"폴리곤 개수: {len(land_gdf)}")
    print(f"전체 경위도 bounds: {land_gdf.total_bounds.tolist()}")
    print(f"empty 개수: {int(land_gdf.geometry.is_empty.sum())}")
    print(f"invalid 개수: {int((~land_gdf.geometry.is_valid).sum())}")

    print("\n=== 2. 기준 좌표 판정 검증 ===")
    all_passed = True
    for name, lat, lon, expected in test_cases:
        actual = is_on_land(lat, lon, land_gdf)
        passed = actual == expected
        all_passed = all_passed and passed
        print(f"[{'PASS' if passed else 'FAIL'}] {name}: lat={lat}, lon={lon} -> actual={actual} (expected={expected})")

    if not all_passed:
        raise AssertionError("기준 좌표 검증 실패: 폴리곤 레이어 또는 버퍼 값을 점검하세요.")


In [ ]:
test_cases = [
    ("테스트 (내륙)", 36.97, 127.94, True),
    ("서울 (내륙)", 37.5665, 126.9780, True),
    ("안양 (내륙)", 37.3943, 126.9568, True),
    ("동해 (해상)", 36.5000, 131.0000, False),
    ("서울 경도 +360 정규화", 37.5665, 126.9780 + 360.0, True),
]
validate_land_gdf(land_gdf_4326, test_cases)

2. RTOFS NetCDF 파일 열기

In [ ]:
ds = xr.open_dataset(file_path[1])

In [ ]:
target_vars = list(ds.data_vars)
print(f"검사 대상 변수: {target_vars}")

In [ ]:
#좌표 정규화
lon_raw = ds["Longitude"].values.flatten()
lat_raw = ds["Latitude"].values.flatten()
# 원시 경도(예: 74.12 ~ 1019.12) -> [0, 360) -> [-180, 180]
lon_360 = np.mod(lon_raw, 360.0)
lon_180 = ((lon_360 + 180.0) % 360.0) - 180.0
# 모든 변수 중 하나라도 유효(non-NaN)한 격자 마스크 생성
combined_valid_mask = np.zeros(len(lon_raw), dtype=bool)

In [ ]:
for var_name in ds.data_vars:
    da = ds[var_name]
    if "MT" in da.dims:
        da = da.isel(MT=0)
    da = da.squeeze()
    val = da.values.flatten()
    if len(val) == len(lon_raw):
        combined_valid_mask |= ~np.isnan(val)

valid_lon_raw = lon_raw[combined_valid_mask]
valid_lon_360 = lon_360[combined_valid_mask]
valid_lon_180 = lon_180[combined_valid_mask]
valid_lat = lat_raw[combined_valid_mask]

print(f"전체 변수 기준 유효 데이터 격자 수: {len(valid_lon_180):,}개")

In [ ]:
#육지 내부 격자 판별 후 바다(비육지) 격자만 추출
points_gdf = gpd.GeoDataFrame(
    {
        "pt_idx": np.arange(len(valid_lon_180)),
        "lon_raw": valid_lon_raw,
        "lon_360": valid_lon_360,
        "lon_180": valid_lon_180,
        "lat": valid_lat,
    },
    geometry=gpd.points_from_xy(valid_lon_180, valid_lat),
    crs="EPSG:4326",
)

In [ ]:
# 육지 폴리곤 안에 들어간 포인트 인덱스 찾기 (중복 매칭 방지 위해 unique)
points_in_land = gpd.sjoin(
    points_gdf[["pt_idx", "geometry"]],
    land_gdf_4326,
    how="inner",
    predicate="within",
)

In [ ]:
#- 결과 출력
land_pt_indices = set(points_in_land["pt_idx"].unique())

# 육지가 아닌 격자 = 바다 영역 데이터
points_in_ocean = points_gdf[~points_gdf["pt_idx"].isin(land_pt_indices)].copy()

print(f"육지 내 포함된 유효 격자 수: {len(land_pt_indices):,}개")
print(f"바다 영역에 포함된 유효 격자 수: {len(points_in_ocean):,}개")
print(points_in_ocean[["lon_raw", "lon_360", "lon_180", "lat"]].head(10))
